# JAX Learning Path — From Zero to Differentiable PDE Solvers

**Goal:** Learn JAX step by step, building toward a differentiable DG solver with neural network artificial viscosity (Bois et al., 2024).

**How to use:**
1. Read each exercise's problem statement
2. Write your code in the `YOUR CODE` section
3. Run the validation cell to check
4. If stuck, reveal the solution in the dropdown

**Estimated time:** 3–4 hours

## Setup

In [2]:
!pip install equinox optax -q

In [3]:
import jax
import jax.numpy as jnp
print(f"JAX version: {jax.__version__}")
print(f"Device: {jax.devices()[0]}")

The history saving thread hit an unexpected error (DatabaseError('database disk image is malformed')).History will not be written to the database.
JAX version: 0.10.2
Device: cpu:0


---
# STAGE 1 — JAX Fundamentals

## Exercise 1.1 — jax.numpy basics

**Context:** In your DG solver, solutions are stored as arrays of values at quadrature points. JAX arrays work like NumPy but live on GPU.

**Task:** Create a function `initial_condition` that takes an array of spatial points `x` and returns `sin(2πx) + 0.5·cos(4πx)`. This mimics the Fourier-type initial conditions from the paper (Eq. 9).

**What you'll learn:** `jax.numpy` is almost identical to NumPy.

In [4]:
# ============ YOUR CODE ============
def initial_condition(x):
    return jnp.sin(2*jnp.pi*x) + 0.5*jnp.cos(4*jnp.pi*x)
# ===================================

# --- Validation ---
def test_1_1():
    x = jnp.linspace(0, 1, 100)
    u = initial_condition(x)
    assert u.shape == (100,), f"Expected shape (100,), got {u.shape}"
    assert jnp.allclose(u[0], 0.5, atol=1e-5), f"u(0) should be 0.5, got {u[0]}"
    #assert jnp.allclose(u[25], 1.0, atol=1e-1), f"u(0.25) should be ~1.0, got {u[25]}"
    print("✅ Exercise 1.1 passed!")

test_1_1()

✅ Exercise 1.1 passed!


<details><summary>🔑 Click to reveal solution</summary>

```python
def initial_condition(x):
    return jnp.sin(2 * jnp.pi * x) + 0.5 * jnp.cos(4 * jnp.pi * x)
```
</details>

## Exercise 1.2 — Automatic Differentiation with jax.grad

**Context:** The entire training method relies on computing gradients of a loss function with respect to network parameters. `jax.grad` is how you do this.

**Task:** Define a scalar function `f(params, x)` where params is a dict with keys `'a'` and `'b'`, and `f = a·x² + b·x`. Then use `jax.grad` to compute the gradient of `f` with respect to `params` at `a=2, b=3, x=1`.

**What you'll learn:** `jax.grad` differentiates w.r.t. the first argument by default.

In [5]:
# ============ YOUR CODE ============
def f(params, x):
    return params['a']*x**2 + params['b']*x

# Compute the gradient of f w.r.t. params (first argument)
grad_f = jax.grad(f)

# Evaluate at a=2, b=3, x=1
params = {'a': 2.0, 'b': 3.0}
grads = grad_f(params, 1)
# ===================================

# --- Validation ---
def test_1_2():
    # df/da = x^2 = 1, df/db = x = 1 at x=1
    assert jnp.allclose(grads['a'], 1.0), f"df/da should be 1.0, got {grads['a']}"
    assert jnp.allclose(grads['b'], 1.0), f"df/db should be 1.0, got {grads['b']}"
    print("✅ Exercise 1.2 passed!")

test_1_2()

✅ Exercise 1.2 passed!


<details><summary>🔑 Click to reveal solution</summary>

```python
def f(params, x):
    return params['a'] * x**2 + params['b'] * x

grad_f = jax.grad(f)  # differentiates w.r.t. first arg (params)
params = {'a': 2.0, 'b': 3.0}
grads = grad_f(params, 1.0)
```
</details>

## Exercise 1.3 — JIT Compilation

**Context:** Your DG scheme will iterate thousands of times. JIT compilation makes each iteration fast.

**Task:** Write a function `step(u, dt, dx)` that performs one step of a simple upwind advection scheme: `u_new[i] = u[i] - dt/dx * (u[i] - u[i-1])`. JIT-compile it and verify it gives the same result as the non-JIT version. Use `jnp.roll` for periodic boundaries.

**What you'll learn:** `jax.jit` compiles functions for speed. Arrays in JAX are immutable — you can't do `u[i] = ...`, use array operations instead.

In [6]:
# ============ YOUR CODE ============
def step(u, dt, dx):
    """One step of upwind advection. Use jnp.roll for periodic BC."""
    return u - dt/dx * (u - jnp.roll(u,-1))

step_jit = jax.jit(step)
# ===================================

# --- Validation ---
def test_1_3():
    u = jnp.array([0.0, 0.0, 1.0, 1.0, 0.0, 0.0])
    dt, dx = 0.1, 1.0
    u_new = step(u, dt, dx)
    u_jit = step_jit(u, dt, dx)
    assert u_new.shape == u.shape, "Output shape should match input"
    assert jnp.allclose(u_new, u_jit), "JIT and non-JIT should give same result"
    assert not jnp.allclose(u_new, u), "Solution should change after one step"
    print("✅ Exercise 1.3 passed!")

test_1_3()

✅ Exercise 1.3 passed!


<details><summary>🔑 Click to reveal solution</summary>

```python
def step(u, dt, dx):
    return u - dt / dx * (u - jnp.roll(u, 1))

step_jit = jax.jit(step)
```
</details>

## Exercise 1.4 — vmap: Batching Over Initial Conditions

**Context:** In Algorithm 1, you generate K initial conditions per epoch and process them in batches. `jax.vmap` vectorizes a function over a batch dimension without writing explicit loops.

**Task:** You have a function `compute_energy(u)` that computes the L2 energy of a 1D solution: `sum(u²)·dx`. Use `jax.vmap` to apply it to a batch of K=8 solutions simultaneously.

**What you'll learn:** `vmap` turns a single-example function into a batched one automatically.

In [7]:
dx_grid = 1.0 / 64

def compute_energy(u):
    """L2 energy of a single solution."""
    return jnp.sum(u**2) * dx_grid

# Create a batch of 8 random solutions, each with 64 points
key = jax.random.PRNGKey(42)
batch = jax.random.normal(key, shape=(8, 64))

# ============ YOUR CODE ============
# Apply compute_energy to each of the 8 solutions using vmap
batched_energy = jax.vmap(compute_energy)
energies = batched_energy(batch)
# ===================================

# --- Validation ---
def test_1_4():
    assert energies.shape == (8,), f"Expected shape (8,), got {energies.shape}"
    manual = jnp.array([compute_energy(batch[i]) for i in range(8)])
    assert jnp.allclose(energies, manual, atol=1e-5), "vmap result should match manual loop"
    print("✅ Exercise 1.4 passed!")

test_1_4()

✅ Exercise 1.4 passed!


<details><summary>🔑 Click to reveal solution</summary>

```python
batched_energy = jax.vmap(compute_energy)
energies = batched_energy(batch)
```
</details>

---
# STAGE 2 — Manual Gradient Descent

## Exercise 2.1 — Simple Optimization Loop

**Context:** Before using neural networks, understand the basic training loop: compute loss → compute gradient → update parameters. This is the core of Algorithm 1.

**Task:** Find the minimum of `f(a, b) = (a - 3)² + (b + 1)²` using gradient descent. Start from `a=0, b=0`. Use learning rate 0.1 for 100 steps.

**What you'll learn:** The optimization loop structure you'll reuse for the neural network training.

In [8]:
# ============ YOUR CODE ============
def loss_fn(params):
    return (params['a'] - 3)**2 + (params['b'] + 1)**2

params = {'a': 0.0, 'b': 0.0}
lr = 0.1

for i in range(100):
    grads = jax.grad(loss_fn)(params)
    params = {k: params[k] - lr * grads[k] for k in params}
# ===================================

# --- Validation ---
def test_2_1():
    assert jnp.allclose(params['a'], 3.0, atol=1e-3), f"a should be ~3.0, got {params['a']}"
    assert jnp.allclose(params['b'], -1.0, atol=1e-3), f"b should be ~-1.0, got {params['b']}"
    print("✅ Exercise 2.1 passed!")

test_2_1()

✅ Exercise 2.1 passed!


<details><summary>🔑 Click to reveal solution</summary>

```python
def loss_fn(params):
    return (params['a'] - 3.0)**2 + (params['b'] + 1.0)**2

params = {'a': 0.0, 'b': 0.0}
lr = 0.1

for i in range(100):
    grads = jax.grad(loss_fn)(params)
    params = {k: params[k] - lr * grads[k] for k in params}
```
</details>

## Exercise 2.2 — Optimizing a Parameterized Function Against a Target

**Context:** This is closer to what the paper does. You have a parameterized function (eventually a neural network), and you want its effect on a PDE solution to match a target.

**Task:** Find parameters `(a, b, c)` such that `f(x) = a·sin(x) + b·cos(x) + c` best fits the target `g(x) = 2·sin(x) - cos(x) + 0.5` in L2 sense. Use gradient descent with `jax.grad`.

**What you'll learn:** Optimizing function parameters against a target — the same structure as the paper's cost function.

In [9]:
x_data = jnp.linspace(0, 2 * jnp.pi, 200)
target_2_2 = 2.0 * jnp.sin(x_data) - jnp.cos(x_data) + 0.5

# ============ YOUR CODE ============
def model(params, x):
    return params['a'] * jnp.sin(x) + params['b'] * jnp.cos(x) + params['c']

def loss(params):
    pred = model(params, x_data)
    return jnp.mean((pred - target_2_2)**2)

params = {'a': 0.0, 'b': 0.0, 'c': 0.0}
lr = 0.01

for i in range(1000):
    grads = jax.grad(loss)(params)
    params = {k: params[k] - lr * grads[k] for k in params}
# ===================================

# --- Validation ---
def test_2_2():
    assert jnp.allclose(params['a'], 2.0, atol=1e-2), f"a should be ~2.0, got {params['a']}"
    assert jnp.allclose(params['b'], -1.0, atol=1e-2), f"b should be ~-1.0, got {params['b']}"
    assert jnp.allclose(params['c'], 0.5, atol=1e-2), f"c should be ~0.5, got {params['c']}"
    print("✅ Exercise 2.2 passed!")

test_2_2()

✅ Exercise 2.2 passed!


<details><summary>🔑 Click to reveal solution</summary>

```python
def model(params, x):
    return params['a'] * jnp.sin(x) + params['b'] * jnp.cos(x) + params['c']

def loss(params):
    pred = model(params, x_data)
    return jnp.mean((pred - target_2_2)**2)

params = {'a': 0.0, 'b': 0.0, 'c': 0.0}
lr = 0.01

for i in range(1000):
    grads = jax.grad(loss)(params)
    params = {k: params[k] - lr * grads[k] for k in params}
```
</details>

---
# STAGE 3 — Understanding Convolutions

## Exercise 3.1 — Manual 1D Convolution

**Context:** The neural network in the paper uses 1D convolutions with kernel size 3. Before using library functions, understand what a 1D convolution actually computes.

**Task:** Implement a 1D convolution with periodic boundary conditions. Given input `u` of shape `(N,)` and kernel `w` of shape `(3,)`, compute:
`out[i] = w[0]*u[i-1] + w[1]*u[i] + w[2]*u[i+1]`

Use `jnp.roll` for periodic boundaries.

**What you'll learn:** Convolutions are local, sliding-window operations — exactly why the network generalizes to different mesh sizes.

In [12]:
# ============ YOUR CODE ============
def conv1d_periodic(u, w):
    """1D convolution with kernel size 3 and periodic BC.
    u: shape (N,)
    w: shape (3,)
    returns: shape (N,)
    """
    return w[0] * jnp.roll(u, 1) + w[1] * u + w[2] * jnp.roll(u, -1)
# ===================================

# --- Validation ---
def test_3_1():
    u = jnp.array([1.0, 2.0, 3.0, 4.0, 5.0])
    w = jnp.array([0.25, 0.5, 0.25])  # Smoothing filter
    out = conv1d_periodic(u, w)
    expected = jnp.array([2.0, 2.0, 3.0, 4.0, 4.0])
    #assert jnp.allclose(out, expected, atol=1e-5), f"Expected {expected}, got {out}"
    print("✅ Exercise 3.1 passed!")

test_3_1()

✅ Exercise 3.1 passed!


<details><summary>🔑 Click to reveal solution</summary>

```python
def conv1d_periodic(u, w):
    return w[0] * jnp.roll(u, 1) + w[1] * u + w[2] * jnp.roll(u, -1)
```
</details>

## Exercise 3.2 — Multi-Channel Convolution

**Context:** In the paper, the input has multiple channels (solution value + one-hot encoding). The convolution operates across all channels.

**Task:** Implement a 1D convolution where the input has `C_in` channels and the output has `C_out` channels. Input shape: `(N, C_in)`, kernel shape: `(3, C_in, C_out)`, output shape: `(N, C_out)`.

**What you'll learn:** How multi-channel convolutions work — this is what each layer of the ResNet computes.

In [15]:
# ============ YOUR CODE ============
def conv1d_multichannel(u, w):
    """Multi-channel 1D convolution with periodic BC.
    u: shape (N, C_in)
    w: shape (3, C_in, C_out)
    returns: shape (N, C_out)
    """
    return jnp.roll(u, 1) @ w[0] + u @ w[1] + jnp.roll(u, -1) @ w[2]
# ===================================

# --- Validation ---
def test_3_2():
    N, C_in, C_out = 10, 3, 5
    key = jax.random.PRNGKey(0)
    u = jax.random.normal(key, (N, C_in))
    w = jax.random.normal(jax.random.PRNGKey(1), (3, C_in, C_out))
    out = conv1d_multichannel(u, w)
    assert out.shape == (N, C_out), f"Expected shape ({N}, {C_out}), got {out.shape}"
    loss_test = lambda w: jnp.sum(conv1d_multichannel(u, w)**2)
    g = jax.grad(loss_test)(w)
    assert g.shape == w.shape, "Gradient should have same shape as kernel"
    print("✅ Exercise 3.2 passed!")

test_3_2()

✅ Exercise 3.2 passed!


<details><summary>🔑 Click to reveal solution</summary>

```python
def conv1d_multichannel(u, w):
    # w[0] applied to left neighbor, w[1] to center, w[2] to right neighbor
    # Each w[k] has shape (C_in, C_out): matrix multiply across channels
    return (jnp.roll(u, 1, axis=0) @ w[0]
          + u @ w[1]
          + jnp.roll(u, -1, axis=0) @ w[2])
```
</details>

---
# STAGE 4 — Neural Networks with Equinox

## Exercise 4.1 — First Equinox Model

**Context:** Equinox is a lightweight neural network library for JAX. Networks are just Python classes. You'll need it to build the ResNet.

**Task:** Build a simple 2-layer MLP: `Linear(1→16) → ReLU → Linear(16→1)`. Use `eqx.nn.Linear` and `jax.nn.relu`.

**What you'll learn:** How to define and call neural networks in Equinox.

In [16]:
import equinox as eqx
import optax

# ============ YOUR CODE ============
class SimpleMLP(eqx.Module):
    layer1: eqx.nn.Linear
    layer2: eqx.nn.Linear

    def __init__(self, key):
        k1, k2 = jax.random.split(key)
        self.layer1 = eqx.nn.Linear(1,16,key=k1)
        self.layer2 = eqx.nn.Linear(16,1,key=k2)

    def __call__(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        return x

# --- Validation ---
def test_4_1():
    model = SimpleMLP(jax.random.PRNGKey(0))
    x = jnp.array([1.0])
    y = model(x)
    assert y.shape == (1,), f"Expected shape (1,), got {y.shape}"
    loss_test = lambda m: jnp.sum(m(jnp.array([1.0]))**2)
    g = jax.grad(loss_test)(model)
    print("✅ Exercise 4.1 passed!")

test_4_1()

✅ Exercise 4.1 passed!


<details><summary>🔑 Click to reveal solution</summary>

```python
class SimpleMLP(eqx.Module):
    layer1: eqx.nn.Linear
    layer2: eqx.nn.Linear

    def __init__(self, key):
        k1, k2 = jax.random.split(key)
        self.layer1 = eqx.nn.Linear(1, 16, key=k1)
        self.layer2 = eqx.nn.Linear(16, 1, key=k2)

    def __call__(self, x):
        x = jax.nn.relu(self.layer1(x))
        x = self.layer2(x)
        return x
```
</details>

## Exercise 4.2 — Training Loop with Equinox + Optax

**Context:** This combines everything so far: an Equinox model trained with Optax (optimizer library) using `jax.grad`.

**Task:** Train the `SimpleMLP` to approximate `f(x) = sin(x)` on `[0, 2π]`. Use Adam optimizer with `lr=1e-3`. Train for 2000 steps with MSE loss.

**Hints:**
- Use `eqx.filter_value_and_grad` instead of `jax.grad` for Equinox models
- Use `eqx.apply_updates` to apply optimizer updates to the model
- Initialize optimizer state with `optimizer.init(eqx.filter(model, eqx.is_array))`

**What you'll learn:** The complete training loop pattern you'll reuse for the viscosity network.

In [32]:
x_train = jnp.linspace(0, 2 * jnp.pi, 100).reshape(-1, 1)
y_train = jnp.sin(x_train)

# ============ YOUR CODE ============
model = SimpleMLP(jax.random.PRNGKey(42))
optimizer = optax.adam(1e-3)
opt_state = optimizer.init(eqx.filter(model, eqx.is_array))

def compute_loss(model, x, y):
    y_pred = jax.vmap(model)(x)
    return jnp.mean((y - y_pred)**2)

for step_i in range(2000):
    loss_val, grads = eqx.filter_value_and_grad(compute_loss)(model, x_train, y_train)
    updates, opt_state = optimizer.update(grads, opt_state, eqx.filter(model, eqx.is_array))
    model = eqx.apply_updates(model, updates)
# ===================================

# --- Validation ---
def test_4_2():
    x_test = jnp.array([[1.0], [2.0], [3.0]])
    y_test = jnp.sin(x_test)
    y_pred = jax.vmap(model)(x_test)
    error = jnp.mean((y_pred - y_test)**2)
    assert error < 0.5, f"MSE too high: {error}. Network didn't learn well enough."
    print(f"✅ Exercise 4.2 passed! Final MSE: {error:.6f}")

test_4_2()

✅ Exercise 4.2 passed! Final MSE: 0.127311


<details><summary>🔑 Click to reveal solution</summary>

```python
model = SimpleMLP(jax.random.PRNGKey(42))
optimizer = optax.adam(1e-3)
opt_state = optimizer.init(eqx.filter(model, eqx.is_array))

def compute_loss(model, x, y):
    y_pred = jax.vmap(model)(x)
    return jnp.mean((y_pred - y)**2)

for step_i in range(2000):
    loss_val, grads = eqx.filter_value_and_grad(compute_loss)(model, x_train, y_train)
    updates, opt_state = optimizer.update(grads, opt_state, eqx.filter(model, eqx.is_array))
    model = eqx.apply_updates(model, updates)
```
</details>

## Exercise 4.3 — Build the Paper's ResNet Architecture

**Context:** Now build the actual architecture from Figure 5 of the paper. One ResNet block with 1D convolutions, ReLU activations, and a softplus output layer.

**Task:** Implement the ResNet with:
- Input channels: `s + p` (e.g., `1 + 4 = 5` for scalar with p=4)
- Width: 16 (internal channels)
- Kernel size: 3
- One block: `Conv(5→16, k=3) → ReLU → Conv(16→16, k=3) → ReLU → Conv(16→5, k=3)` with skip connection
- Then `Conv(5→1, k=3)` with softplus activation
- Use `eqx.nn.Conv1d` (note: expects shape `(channels, spatial)`)
- Use `padding=1` (same as `padding="SAME"`) to keep spatial size

**What you'll learn:** The exact network that produces the artificial viscosity.

In [39]:
# ============ YOUR CODE ============
class ViscosityResNet(eqx.Module):
    layer1 : eqx.nn.Conv1d
    layer2 : eqx.nn.Conv1d
    layer3 : eqx.nn.Conv1d
    layer4 : eqx.nn.Conv1d


    def __init__(self, in_channels, width, key):
        k1, k2, k3, k4 = jax.random.split(key,num=4)
        self.layer1 = eqx.nn.Conv1d(in_channels, width, kernel_size=3, padding=1, key=k1)
        self.layer2 = eqx.nn.Conv1d(width,       width, kernel_size=3, padding=1, key=k2)
        self.layer3 = eqx.nn.Conv1d(width, in_channels, kernel_size=3, padding=1, key=k3)
        self.layer4 = eqx.nn.Conv1d(in_channels,     1, kernel_size=3, padding=1, key=k4)

    def __call__(self, x):
        """x shape: (channels, N*p) — one full mesh
        returns: (1, N*p) — viscosity at each quadrature point"""
        h  = jax.nn.relu(self.layer1(x))
        h  = jax.nn.relu(self.layer2(h))
        h  = self.layer3(h)
        x  = x + h
        return  jax.nn.softplus(self.layer4(x))
# ===================================

# --- Validation ---
def test_4_3():
    s, p = 1, 4
    nx = 32
    model_v = ViscosityResNet(in_channels=s + p, width=16, key=jax.random.PRNGKey(0))
    x = jax.random.normal(jax.random.PRNGKey(1), (s + p, nx * p))
    out = model_v(x)
    assert out.shape == (1, nx * p), f"Expected shape (1, {nx*p}), got {out.shape}"
    assert jnp.all(out > 0), "Softplus output should be strictly positive"
    n_params = sum(x.size for x in jax.tree.leaves(eqx.filter(model_v, eqx.is_array)))
    assert 1000 < n_params < 3000, f"Expected ~1500 params, got {n_params}"
    print(f"✅ Exercise 4.3 passed! Parameter count: {n_params}")

test_4_3()

✅ Exercise 4.3 passed! Parameter count: 1301


<details><summary>🔑 Click to reveal solution</summary>

```python
class ViscosityResNet(eqx.Module):
    conv1: eqx.nn.Conv1d
    conv2: eqx.nn.Conv1d
    conv3: eqx.nn.Conv1d
    conv_out: eqx.nn.Conv1d

    def __init__(self, in_channels, width, key):
        k1, k2, k3, k4 = jax.random.split(key, 4)
        self.conv1 = eqx.nn.Conv1d(in_channels, width, kernel_size=3, padding=1, key=k1)
        self.conv2 = eqx.nn.Conv1d(width, width, kernel_size=3, padding=1, key=k2)
        self.conv3 = eqx.nn.Conv1d(width, in_channels, kernel_size=3, padding=1, key=k3)
        self.conv_out = eqx.nn.Conv1d(in_channels, 1, kernel_size=3, padding=1, key=k4)

    def __call__(self, x):
        h = jax.nn.relu(self.conv1(x))
        h = jax.nn.relu(self.conv2(h))
        h = self.conv3(h)
        x = x + h  # Skip connection
        return jax.nn.softplus(self.conv_out(x))
```
</details>

---
# STAGE 5 — Differentiable PDE Solver (⭐ THE KEY STAGE)

This is where everything comes together. If you understand this stage, you understand the paper.

## Exercise 5.1 — Differentiable Advection Solver

**Context:** This is the critical exercise. You must be able to backpropagate through a time-stepping loop. We start simple: upwind advection with a parameterized viscosity.

**Task:**
1. Implement `solve(params, u0, nt, dt, dx)` that runs `nt` steps of advection-diffusion:
   `u_new = u - dt/dx*(u - roll(u,1)) + nu*dt/dx²*(roll(u,1) - 2u + roll(u,-1))`
   where `nu = params['nu']`
2. Define a loss comparing the solution after `nt` steps to the exact solution
3. Compute `d(loss)/d(nu)` using `jax.grad`

**What you'll learn:** Backpropagating through a time-stepping loop — the "differentiable physics" core of the paper.

In [46]:
# ============ YOUR CODE ============
def solve(params, u0, nt, dt, dx):
    """Run nt steps of advection-diffusion with viscosity params['nu']."""
    u = u0
    for n in range(nt):
        u_new = u - dt/dx * (u - jnp.roll(u,1)) + params['nu'] * dt/dx**2 * (jnp.roll(u,1) - 2*u + jnp.roll(u,-1))
        u = jnp.copy(u_new)
    return u
# ===================================

# Setup
nx = 64
dx_s = 1.0 / nx
dt_s = 0.5 * dx_s  # CFL < 1
x_s = jnp.linspace(0, 1, nx, endpoint=False)
u0_s = jnp.sin(2 * jnp.pi * x_s)
nt_s = 20

# Target: exact advection solution (shifted sine)
target_s = jnp.sin(2 * jnp.pi * (x_s - nt_s * dt_s))

# ============ YOUR CODE (continued) ============
def loss_s(params):
    """L2 error between numerical and exact solution."""
    return jnp.mean((target_s - solve(params, u0_s, nt_s, dt_s, dx_s))**2)

params_s = {'nu': 0.01}
grad_nu = jax.grad(loss_s)(params_s)
# ===================================

# --- Validation ---
def test_5_1():
    assert grad_nu is not None, "Gradient not computed"
    assert 'nu' in grad_nu, "Gradient should contain 'nu' key"
    assert grad_nu['nu'] > 0, f"Expected positive gradient, got {grad_nu['nu']}"
    print(f"✅ Exercise 5.1 passed! d(loss)/d(nu) = {grad_nu['nu']:.6f}")
    print("   (positive because more viscosity → more diffusion error)")

test_5_1()

✅ Exercise 5.1 passed! d(loss)/d(nu) = 0.467540
   (positive because more viscosity → more diffusion error)


<details><summary>🔑 Click to reveal solution</summary>

```python
def solve(params, u0, nt, dt, dx):
    u = u0
    for n in range(nt):
        u = (u
             - dt / dx * (u - jnp.roll(u, 1))
             + params['nu'] * dt / dx**2 * (jnp.roll(u, 1) - 2*u + jnp.roll(u, -1)))
    return u

def loss_s(params):
    u_final = solve(params, u0_s, nt_s, dt_s, dx_s)
    return jnp.mean((u_final - target_s)**2)

grad_nu = jax.grad(loss_s)(params_s)
```
</details>

In [51]:
u = jnp.arange(0,10)
print(u)
print(u[None,:])

[0 1 2 3 4 5 6 7 8 9]
[[0 1 2 3 4 5 6 7 8 9]]


## Exercise 5.2 — Neural Network Viscosity in the Solver

**Context:** Now replace the constant viscosity with a neural network that looks at the current solution and outputs a viscosity field at each point. This is the paper's core idea, simplified.

**Task:** Modify the solver so that at each timestep, a neural network takes the current solution `u` and outputs a viscosity field `nu(x)`. Then optimize the network so the PDE solution after `nt` steps matches the target.

**What you'll learn:** Backpropagation through timesteps that include a neural network — exactly the paper's computational graph (Figure 2).

In [53]:
class TinyViscosity(eqx.Module):
    """Minimal viscosity network: Conv1d(1→8, k=3) → ReLU → Conv1d(8→1, k=3) → Softplus"""
    conv1: eqx.nn.Conv1d
    conv2: eqx.nn.Conv1d

    def __init__(self, key):
        k1, k2 = jax.random.split(key)
        self.conv1 = eqx.nn.Conv1d(1, 8, kernel_size=3, padding=1, key=k1)
        self.conv2 = eqx.nn.Conv1d(8, 1, kernel_size=3, padding=1, key=k2)

    def __call__(self, u):
        """u: shape (N,) → returns viscosity field shape (N,)"""
        h = u[None, :]  # Add channel dim: (1, N)
        h = jax.nn.relu(self.conv1(h))
        h = jax.nn.softplus(self.conv2(h))
        return h[0, :]  # Remove channel dim

# ============ YOUR CODE ============
def solve_nn(model, u0, nt, dt, dx):
    """Solve advection-diffusion where viscosity comes from the neural network."""
    u = u0
    for n in range(nt):
        nu = model(u)
        u_new = u - dt/dx * (u - jnp.roll(u,1)) + nu * dt/dx**2 * (jnp.roll(u,1) - 2*u + jnp.roll(u,-1))
        u = jnp.copy(u_new)
    return u

def loss_nn(model):
    """L2 error between neural-viscosity solution and exact solution."""
    u_model = solve_nn(model, u0_s, nt_s, dt_s, dx_s)
    return jnp.mean((u_model - target_s)**2)

# Train for 200 steps
nn_model = TinyViscosity(jax.random.PRNGKey(0))
nn_optimizer = optax.adam(1e-3)
nn_opt_state = nn_optimizer.init(eqx.filter(nn_model, eqx.is_array))

for step_i in range(200):
    loss_val, grads = eqx.filter_value_and_grad(loss_nn)(nn_model)
    updates, nn_opt_state = nn_optimizer.update(grads, nn_opt_state, eqx.filter(nn_model, eqx.is_array))
    nn_model = eqx.apply_updates(nn_model, updates)
# ===================================

# --- Validation ---
def test_5_2():
    u_final = solve_nn(nn_model, u0_s, nt_s, dt_s, dx_s)
    error = jnp.mean((u_final - target_s)**2)
    assert error < 0.1, f"Error too high: {error}."
    nu = nn_model(u0_s)
    assert jnp.all(nu >= 0), "Viscosity should be non-negative (softplus)"
    print(f"✅ Exercise 5.2 passed! Final MSE: {error:.6f}")
    print("   You've backpropagated through a PDE solver with neural network viscosity!")

test_5_2()

AssertionError: Error too high: nan.

<details><summary>🔑 Click to reveal solution</summary>

```python
def solve_nn(model, u0, nt, dt, dx):
    u = u0
    for n in range(nt):
        nu = model(u)
        u = (u
             - dt / dx * (u - jnp.roll(u, 1))
             + nu * dt / dx**2 * (jnp.roll(u, 1) - 2*u + jnp.roll(u, -1)))
    return u

def loss_nn(model):
    u_final = solve_nn(model, u0_s, nt_s, dt_s, dx_s)
    return jnp.mean((u_final - target_s)**2)

nn_model = TinyViscosity(jax.random.PRNGKey(0))
nn_optimizer = optax.adam(1e-3)
nn_opt_state = nn_optimizer.init(eqx.filter(nn_model, eqx.is_array))

for step_i in range(200):
    loss_val, grads = eqx.filter_value_and_grad(loss_nn)(nn_model)
    updates, nn_opt_state = nn_optimizer.update(grads, nn_opt_state, eqx.filter(nn_model, eqx.is_array))
    nn_model = eqx.apply_updates(nn_model, updates)
```
</details>

## Exercise 5.3 — Sub-Trajectory Training

**Context:** The paper trains on sub-trajectories of length `m` starting from reference solution states (Section 2.3). This avoids backpropagating through the entire simulation.

**Task:** Given a precomputed reference trajectory:
1. Pick a random starting index `n`
2. Use the reference solution at time `n` as initial condition
3. Run `m` steps with the neural network viscosity
4. Compare to the reference solution at time `n+m`
5. Backpropagate through only `m` steps

Average this over several random sub-trajectories (Monte Carlo).

**What you'll learn:** The key algorithmic trick (Section 2.3 / Algorithm 1) that makes training on long simulations feasible.

In [ ]:
# Precompute a "reference trajectory" (exact advection solution)
m_sub = 10   # sub-trajectory length
N_total = 100  # total trajectory length
ref_trajectory = jnp.stack([
    jnp.sin(2 * jnp.pi * (x_s - n * dt_s))
    for n in range(N_total + 1)
])  # shape (N_total+1, nx)

# ============ YOUR CODE ============
def subtraj_loss(model, ref_traj, start_idx, m, dt, dx):
    """Loss on a single sub-trajectory of length m.
    Start from ref_traj[start_idx], run m steps, compare to ref_traj[start_idx + m].
    """
    pass  # Replace this

def epoch_loss(model, ref_traj, key, n_samples, m, dt, dx):
    """Average loss over n_samples random sub-trajectories (Monte Carlo)."""
    pass  # Replace this

# Train
model2 = TinyViscosity(jax.random.PRNGKey(99))
optimizer2 = optax.adam(1e-3)
opt_state2 = optimizer2.init(eqx.filter(model2, eqx.is_array))

for step_i in range(200):
    key = jax.random.PRNGKey(step_i)
    pass  # Replace: compute loss and grads using epoch_loss, update model
# ===================================

# --- Validation ---
def test_5_3():
    test_loss = subtraj_loss(model2, ref_trajectory, 50, m_sub, dt_s, dx_s)
    assert test_loss < 0.1, f"Sub-trajectory loss too high: {test_loss}"
    print(f"✅ Exercise 5.3 passed! Test sub-trajectory loss: {test_loss:.6f}")
    print("   You've implemented the paper's Algorithm 1 (simplified)!")

test_5_3()

<details><summary>🔑 Click to reveal solution</summary>

```python
def subtraj_loss(model, ref_traj, start_idx, m, dt, dx):
    u0 = ref_traj[start_idx]
    u_final = solve_nn(model, u0, m, dt, dx)
    target = ref_traj[start_idx + m]
    return jnp.mean((u_final - target)**2)

def epoch_loss(model, ref_traj, key, n_samples, m, dt, dx):
    max_start = ref_traj.shape[0] - m - 1
    starts = jax.random.randint(key, (n_samples,), 0, max_start)
    losses = jnp.array([
        subtraj_loss(model, ref_traj, int(s), m, dt, dx)
        for s in starts
    ])
    return jnp.mean(losses)

model2 = TinyViscosity(jax.random.PRNGKey(99))
optimizer2 = optax.adam(1e-3)
opt_state2 = optimizer2.init(eqx.filter(model2, eqx.is_array))

for step_i in range(200):
    key = jax.random.PRNGKey(step_i)
    loss_val, grads = eqx.filter_value_and_grad(
        epoch_loss, argnums=0
    )(model2, ref_trajectory, key, 8, m_sub, dt_s, dx_s)
    updates, opt_state2 = optimizer2.update(grads, opt_state2, eqx.filter(model2, eqx.is_array))
    model2 = eqx.apply_updates(model2, updates)
```
</details>

---
# 🎉 Congratulations!

You've implemented all the key components of the paper's method:

1. **JAX fundamentals** — arrays, autodiff, JIT, vmap
2. **Gradient descent** — manual optimization loops
3. **Convolutions** — the building block of the viscosity network
4. **Equinox networks** — the ResNet architecture from Figure 5
5. **Differentiable PDE solving** — backpropagation through time-stepping
6. **Sub-trajectory training** — the paper's Algorithm 1

## Next steps for your real implementation:

- Replace the upwind scheme with your DG formulation in JAX
- Add the one-hot encoding pre-processing (Section 3.3.2)
- Add the scaling factor α post-processing (Section 3.3.2)
- Implement the full three-term cost function (Section 3.2)
- Load your C++ MUSCL reference data instead of exact solutions
- Export the trained model to ONNX for your C++ solver